# Backpropagation and PyTorch

- Thursdays, 3:30–6:00 PM · ICC 103
- Week 3 of 14
- **Quiz 2 is at the end of class today**, covering Week 2 and Lab 2

## Agenda

1. **Why backpropagation exists**: what Lab 2 cost, priced at scale
2. **The chain rule**, in the form deep learning actually uses it
3. **Computational graphs**: the forward pass, recorded
4. **Reverse-mode autodiff**, worked by hand and then in PyTorch
5. **From a scalar to a layer**, and why nobody stores a Jacobian
6. **PyTorch autograd**: tensors, `.backward()`, `nn.Module`
7. **The first training loop**

Spotlights first. Quiz 2 in the last 20 minutes.

## Lab 2 recap

You trained a $1 \to 16 \to 1$ tanh network on a noisy sine with **no
backpropagation**. Finite differences, 400 steps of gradient descent:

| | |
|---|---|
| Parameters | 49 |
| Forward passes per gradient | 50 |
| Total forward passes | 20,001 |
| Cost, start to end | 1.3256 → 0.0190 |
| A straight line, for comparison | 0.1736 |
| The noise floor | 0.0113 |

The fit was real. It beat a straight line by an order of magnitude and landed
near the noise floor. No library on earth trains a network this way.

## Cost of one gradient step {.smaller}

[![](images/fig-cost-scaling.png){width=1150 .dw95}](images/fig-cost-scaling.png){target="_blank" .zoom}

Left is measured on this laptop: the gap widens from 150x to over 1,600x across
that range. Right is not measured, because it does not need to be. Finite
differences take **exactly** $n+1$ forward passes for $n$ parameters. At one
millisecond each, ResNet-50 needs 7 hours and GPT-3 needs 5.5 years, for **one**
step of gradient descent.

## Limits of analytical solutions

Week 1 found the least-squares solution in closed form. That does not generalize:

1. **Scale.** 175 billion parameters is 175 billion coupled equations
2. **Nonlinearity.** $\tanh$, ReLU and softmax compose into transcendental
   equations with no algebraic solution
3. **Non-convexity.** Many local minima and many saddle points, so "the"
   solution is not a well-posed request
4. **No closed form.** Setting the gradient to zero gives you a system nobody
   can solve symbolically

So we iterate. Iterating needs a gradient at the current parameters, and it
needs one **per step**. The gradient is the thing that has to be cheap.

## Three ways to get a gradient

| | Exact? | Cost for $n$ parameters | Works on a network? |
|---|---|---|---|
| Symbolic, by hand | yes | your whole afternoon, once | only for tiny ones |
| Finite differences | **no**, approximate | $n+1$ forward passes | yes, and too slowly |
| Reverse-mode autodiff | **yes**, to floating point | 1 forward + 1 backward | this is what we use |

Finite differences are worse on both axes. They are approximate **and** slow.
The step size $\varepsilon$ has to be small enough to be accurate and large
enough to survive floating-point cancellation, and those two demands fight.

Autodiff has no $\varepsilon$. It is exact arithmetic on the derivative rules
you already know.

## Automatic differentiation

**Automatic differentiation** computes partial derivatives of an arbitrary
computer program by applying the chain rule to every elementary operation the
program ran.

It works because every line of a forward pass is one of a short list:

- **Operations:** add, subtract, multiply, divide
- **Functions:** $\exp$, $\log$, $\sin$, $\tanh$, $\max$

Each has a derivative you learned in Calc I. Autodiff records which ones ran,
in what order, on what values, and then walks that record backward.

PyTorch, TensorFlow and JAX are all autodiff engines with a neural-network
library attached.

## Backpropagation

**Backpropagation** is reverse-mode autodiff applied to a neural network. It
answers, for every weight and every bias in the model:

$$\frac{\partial \mathcal{L}}{\partial w} = \; ? \qquad
  \frac{\partial \mathcal{L}}{\partial b} = \; ?$$

after **one** forward pass and **one** backward pass, regardless of how many
parameters there are.

::: {.callout-note}
## The name is narrower than the idea
Autodiff is the general procedure and predates deep learning. Backpropagation
is its application to networks. In practice people use the words
interchangeably, and you should know which one is the superset.
:::

## Loss and cost {.smaller}

These get used interchangeably. They are not the same thing, and today the
distinction matters because the backward pass starts at one of them.

:::: {.columns}
::: {.column width="48%"}
**Loss** $\ell_i$: the error on a **single** example.

$$\ell_i = (y_i - \hat{y}_i)^2$$

$$\ell_i = -\big(y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\big)$$
:::
::: {.column width="48%"}
**Cost** $\mathcal{L}$: the reduction of those losses over a batch of $N$.

$$\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N} \ell_i$$

The "mean" in *mean* squared error is what turns a loss into a cost.
:::
::::

The optimizer only ever sees a **scalar**. `loss.backward()` in PyTorch is
called on the cost, and that scalar is why reverse mode is the right choice.

# The chain rule

- One rule, applied a few million times
- The multivariable version is the one that matters

## Chain rule for composite functions

A network is a composition of simple functions, so the chain rule is the entire
mathematical content of backpropagation. There is no limit to the depth of the
composition:

$$\frac{d}{dx}\big[f(g(h(x)))\big]
  = f'(g(h(x)))\; g'(h(x))\; h'(x)$$

Take $f(x) = e^x$, $g(x) = \sin x$, $h(x) = x^2$, so
$f(g(h(x))) = e^{\sin(x^2)}$:

$$\frac{d}{dx}\big[e^{\sin(x^2)}\big]
  = e^{\sin(x^2)} \cdot \frac{d}{dx}\big[\sin(x^2)\big]
  = e^{\sin(x^2)} \cos(x^2) \cdot \frac{d}{dx}\big[x^2\big]
  = 2x\, e^{\sin(x^2)} \cos(x^2)$$

Read it right to left and you have just done a backward pass on a three-node
graph.

## Multivariable chain rule

Networks have thousands to billions of parameters, and a parameter early in the
network reaches the cost through **many** downstream paths. The single-variable
rule is not enough.

Let $z = f(x, y)$ where $x = g(w, b)$ and $y = h(w, b)$. Then

$$\frac{\partial z}{\partial w}
  = \frac{\partial z}{\partial x}\frac{\partial x}{\partial w}
  + \frac{\partial z}{\partial y}\frac{\partial y}{\partial w}$$

$$\frac{\partial z}{\partial b}
  = \frac{\partial z}{\partial x}\frac{\partial x}{\partial b}
  + \frac{\partial z}{\partial y}\frac{\partial y}{\partial b}$$

Think of $x$ and $y$ as two hidden units that both feed a unit downstream.
$w$ affects the result through both, so **both paths are counted, and they add.**

## Multivariable chain rule, worked

Let $z = e^{xy}$ where $x = 2w + b$ and $y = \sin w + \cos 2b$.

The local derivatives are
$\partial z/\partial x = y e^{xy}$, $\partial z/\partial y = x e^{xy}$,
$\partial x/\partial w = 2$, $\partial y/\partial w = \cos w$.

$$\frac{\partial z}{\partial w}
  = \underbrace{y e^{xy}}_{\text{path through } x} \cdot 2
  + \underbrace{x e^{xy}}_{\text{path through } y} \cdot \cos w$$

$$\frac{\partial z}{\partial b}
  = y e^{xy} \cdot 1
  + x e^{xy} \cdot (-2 \sin 2b)$$

Nothing here is hard. It is bookkeeping. Autodiff exists because the bookkeeping
does not stay manageable past about six nodes.

## Chain rule along a network path {.smaller}

[![](images/2024-03-12-17-07-44.png){width=735 .dw70}](images/2024-03-12-17-07-44.png){target="_blank" .zoom}

The bottom rows are the part to study: each factor written once as a **formula**
and once as its **value**, with $\partial L / \partial w$ the product of the
row, $(1)(2)(1)(0.187)(-16) \approx -6$.

## Generalized chain rule

The statement with no ceiling on the number of functions or variables, which is
the shape a real network takes.

Let $z = f(x_1, x_2, \ldots, x_m)$ be differentiable in $m$ variables, and let
each $x_i = x_i(t_1, t_2, \ldots, t_n)$ be differentiable in $n$ variables.
Then for any $j$:

$$\frac{\partial z}{\partial t_j}
  = \frac{\partial z}{\partial x_1}\frac{\partial x_1}{\partial t_j}
  + \frac{\partial z}{\partial x_2}\frac{\partial x_2}{\partial t_j}
  + \cdots
  + \frac{\partial z}{\partial x_m}\frac{\partial x_m}{\partial t_j}$$

Read $z$ as the cost, $t_j$ as one weight, and the $x_i$ as every activation
that weight touches. That sum is the backward pass.

## Gradient notation

The **gradient** of a function collects all of its partial derivatives into one
object, written $\nabla f$.

For $z = f(x_1, \ldots, x_m)$:

$$\nabla_{\mathbf{x}} f =
  \begin{bmatrix}
  \partial f / \partial x_1 \\
  \partial f / \partial x_2 \\
  \vdots \\
  \partial f / \partial x_m
  \end{bmatrix}$$

In deep learning we want the gradient of the cost with respect to the
**parameters**, written $\nabla_{\theta} \mathcal{L}$, where $\theta$ is the
whole parameter set. Lab 2 called that flat vector `theta` and it had 49 entries.

## Gradient of the cost with respect to a weight matrix {.smaller}

[![](images/fig-gradient-shapes.png){width=1050 .dw70}](images/fig-gradient-shapes.png){target="_blank" .zoom}

For $\mathbf{W}$ of shape $(m, n)$, the gradient
$\nabla_{\mathbf{W}} \mathcal{L}$ is also $(m, n)$:

$$\nabla_{\mathbf{W}} \mathcal{L} =
  \begin{bmatrix}
  \partial \mathcal{L}/\partial w_{1,1} & \cdots & \partial \mathcal{L}/\partial w_{1,n} \\
  \vdots & \ddots & \vdots \\
  \partial \mathcal{L}/\partial w_{m,1} & \cdots & \partial \mathcal{L}/\partial w_{m,n}
  \end{bmatrix}$$

This is worth pausing on, because it is what makes `theta = theta - lr * grad`
a legal line of code.

# Computational graphs

- Nodes are values, edges are operations
- The forward pass builds the graph; the backward pass walks it

## Computational graphs as directed acyclic graphs

[![](images/computational-graph-sigmoid.png){width=735 .dw70}](images/computational-graph-sigmoid.png){target="_blank" .zoom}

A network's computational graph is a **directed acyclic graph**:

- Each node is a value that some operation produced
- Each edge is an operation with a known derivative
- **Acyclic** means no cycles, which is what makes a single backward sweep
  well-defined
- Nodes can be as primitive or as coarse as you like. $\sigma(z)$ can be four
  nodes or one

## Graph of an expression {.smaller}

:::: {.columns}
::: {.column width="42%"}
[![](images/computational-graph-2.png){width=420 .dw40}](images/computational-graph-2.png){target="_blank" .zoom}
:::
::: {.column width="58%"}
Before any numbers, the graph is just the structure of the expression. Reading
it left to right: $w$ and $x$ meet at a multiply, the result meets $v$ at an
add, and that meets $-1$ at a multiply.

Nothing here is specific to neural networks. Any expression built from
differentiable pieces has a graph like this, which is why autodiff works on
ordinary Python code and not only on models.
:::
::::

## Forward pass on a graph {.smaller}

[![](images/kahns-algorithm.png){width=714 .dw68}](images/kahns-algorithm.png){target="_blank" .zoom}

Three inputs and four operations. $A \times B = C$, then $D + C = F$, then
$F + A = G$, then $G \times B = H$. The **green** numbers are the forward
values; ignore the orange ones for now.

The forward pass does two things, and the second is the one people forget: it
computes the values, **and it records the operations and their inputs.** That
record is what the backward pass consumes, and it is why a forward pass under
`torch.no_grad()` uses less memory.

## The chain rule on a graph

The rule becomes local. At a node $v$ with parent $u$:

$$\frac{\partial \mathcal{L}}{\partial u}
  = \underbrace{\frac{\partial \mathcal{L}}{\partial v}}_{\text{arrives from the right}}
    \cdot
    \underbrace{\frac{\partial v}{\partial u}}_{\text{one derivative, computed here}}$$

Each node needs to know exactly two things:

1. The gradient handed to it by its children, called the **upstream gradient**
2. The derivative of its own operation, called the **local gradient**

Multiply them, pass the product to the parents. No node knows the size of the
network it is in, and none of them needs to.

## Local derivatives at a node

[![](images/backpropagation-node-types.png){width=945 .dw90}](images/backpropagation-node-types.png){target="_blank" .zoom}

- **Add.** $z = a + b$ has $\partial z/\partial a = 1$, so the upstream
  gradient is **copied** to every input
- **Multiply.** $z = ab$ has $\partial z/\partial a = b$, so the gradients
  **swap**: $a$ receives $b \cdot \partial \mathcal{L}/\partial z$
- **Max.** The gradient **routes** to whichever input won, and the other gets
  zero. ReLU is $\max(0, z)$, so a negative pre-activation gets nothing

Three rules cover most of a feedforward network. What does a **min** gate do?
What about concatenation?

## Backward pass on a graph {.smaller}

[![](images/kahns-algorithm.png){width=714 .dw68}](images/kahns-algorithm.png){target="_blank" .zoom}

Same graph, now read the **orange** numbers, which are the gradients of $H$
with respect to each edge. They are produced right to left, starting from
$\partial H / \partial H = 1$ at the output.

Every orange number is an upstream gradient times one local derivative. At
$H = G \times B$, for instance, $G$ receives $1 \cdot B = 3$ and $B$ receives
$1 \cdot G = 13$; the multiply node swaps its inputs.

## Gradients sum over paths

[![](images/fig-fork-adds.png){width=1050 .dw90}](images/fig-fork-adds.png){target="_blank" .zoom}

$A$ reaches the output through $C$ and through $G$, so the multivariable chain
rule says count both and add: $-9 + (-3) = -12$.

This is not a detail. **It is the reason PyTorch accumulates into `.grad`
instead of overwriting it,** and therefore the reason you have to call
`zero_grad()`. We will trip over that on purpose later.

## A sigmoid expanded into primitive nodes {.smaller}

[![](images/backpropagation-single-node-sigmoid.png){width=725 .dw68}](images/backpropagation-single-node-sigmoid.png){target="_blank" .zoom}

Written out, $\sigma(z) = 1/(1 + e^{-z})$ is four operations: negate, exponentiate,
add one, take the reciprocal. Each is a node with its own local derivative, and
backprop runs through all four.

## The same sigmoid as one node {.smaller}

[![](images/backpropagation-single-node-sigmoid-simplified.png){width=725 .dw60}](images/backpropagation-single-node-sigmoid-simplified.png){target="_blank" .zoom}

Because $\sigma$ has a derivative we know in closed form, those four nodes
collapse into one:

$$\sigma'(z) = \sigma(z)\big(1 - \sigma(z)\big)$$

Same gradient, one node instead of four. The forward pass already computed
$\sigma(z)$, so the backward pass gets the local gradient for free from a value
it has cached. This is the general pattern: autodiff engines trade memory for
speed by keeping forward values around.

## Execution order and topological sorting

A node cannot compute its gradient until **every** child has handed one over.
Look at $A$ in the last figure: process it too early and you get $-3$ instead of
$-12$.

So the engine needs an ordering of the nodes where no node comes before something
it depends on. That ordering is a **topological sort**, and it is not unique. For
the graph below, both of these are valid:

$$[0, 1, 2, 3, 4, 5, 6, 7] \qquad [1, 0, 4, 3, 7, 6, 5, 2]$$

[![](images/topological-sort.png){width=420 .dw40}](images/topological-sort.png){target="_blank" .zoom}

## Kahn's algorithm

Repeatedly take a node with no remaining edges into it, emit it, and delete its
edges. If edges are left over at the end, the graph had a cycle and there is no
valid order.

We will walk the next eight slides one at a time. Watch two things:

- the **queue** on the left, which holds every node that is ready to be emitted
- the **topological order** along the bottom, which fills left to right

We start at the output node and work back toward the inputs, so the order this
produces is exactly the order `loss.backward()` visits nodes in.

## Topological sort: the starting queue

[![](images/kahns-algorithm-00.png){width=880 .dw85}](images/kahns-algorithm-00.png){target="_blank" .zoom}

## Topological sort: step 1

[![](images/kahns-algorithm-01.png){width=880 .dw85}](images/kahns-algorithm-01.png){target="_blank" .zoom}

## Topological sort: step 2

[![](images/kahns-algorithm-02.png){width=880 .dw85}](images/kahns-algorithm-02.png){target="_blank" .zoom}

## Topological sort: step 3

[![](images/kahns-algorithm-03.png){width=880 .dw85}](images/kahns-algorithm-03.png){target="_blank" .zoom}

## Topological sort: step 4

[![](images/kahns-algorithm-04.png){width=880 .dw85}](images/kahns-algorithm-04.png){target="_blank" .zoom}

## Topological sort: step 5

[![](images/kahns-algorithm-05.png){width=880 .dw85}](images/kahns-algorithm-05.png){target="_blank" .zoom}

## Topological sort: step 6

[![](images/kahns-algorithm-06.png){width=880 .dw85}](images/kahns-algorithm-06.png){target="_blank" .zoom}

## Topological sort: the finished order

[![](images/kahns-algorithm-07.png){width=880 .dw85}](images/kahns-algorithm-07.png){target="_blank" .zoom}

## Topological sort, animated

[![](images/kahns-algorithm.gif){width=880 .dw85}](images/kahns-algorithm.gif){target="_blank" .zoom}

The same eight steps, run end to end.

## Kahn's algorithm in pseudocode {.smaller}

```
L <- empty list that will hold the sorted elements
S <- set of all nodes with no incoming edge

while S is not empty:
    remove a node n from S
    add n to L
    for each node m with an edge e from n to m:
        remove edge e from the graph
        if m has no other incoming edges:
            insert m into S

if the graph still has edges:
    return error          # at least one cycle
else:
    return L              # a topologically sorted order
```

You will not implement this. You should know that it exists, that it is why the
graph has to be acyclic, and that a recurrent network gets around the "acyclic"
requirement by unrolling in time, which is Week 9.

# Reverse-mode autodiff

- One scalar example, worked by hand
- Then the identical example in PyTorch, so the API is not a mystery

## Scalar regression example {.smaller}

Take $x = 3$, $w = 2$, $b = 4$, a ReLU, and squared error against $y = 12$:

$$z_1 = wx = 6, \quad
  z_2 = z_1 + b = 10, \quad
  a = \mathrm{ReLU}(z_2) = 10, \quad
  \ell = (a - 12)^2 = 4$$

Four numbers and four recorded operations. This is a single neuron, which is
exactly the object Week 2 built. Now walk it backward, starting from
$\partial \ell / \partial \ell = 1$:

$$\frac{\partial \ell}{\partial a} = 2(a - y) = -4,
  \quad
  \frac{\partial \ell}{\partial z_2} = -4 \cdot \mathbb{1}[z_2 > 0] = -4,
  \quad
  \frac{\partial \ell}{\partial w} = -4x = -12,
  \quad
  \frac{\partial \ell}{\partial b} = -4$$

ReLU passed the gradient through because $z_2 > 0$. Had $z_2$ been negative,
every gradient to its left would be exactly zero and this neuron would learn
nothing from this example.

## Scalar regression graph, both passes {.smaller}

[![](images/scalar-backpropagation-regression.png){width=945 .dw90}](images/scalar-backpropagation-regression.png){target="_blank" .zoom}

Green is the forward pass, orange the backward pass, and every node carries the
chain rule that produced its number. ReLU and MSE, with their derivatives, are
in the top right.

## The same graph in PyTorch

`requires_grad=True` tells autograd to record. `retain_grad()` asks it to keep
the gradient at an intermediate node, which it normally discards.

In [1]:
#| echo: true
import torch
import torch.nn as nn

x = torch.tensor([3.0], requires_grad=True)
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([4.0], requires_grad=True)
y = torch.tensor([12.0])

z1 = w * x;                z1.retain_grad()
z2 = z1 + b;               z2.retain_grad()
a = torch.relu(z2);        a.retain_grad()
loss = nn.MSELoss()(a, y)

loss.backward()

print(f"forward    z1={z1.item():5.1f}  z2={z2.item():5.1f}  "
      f"a={a.item():5.1f}  loss={loss.item():5.1f}")
print()
for name, t in [("a", a), ("z2", z2), ("z1", z1), ("b", b), ("w", w), ("x", x)]:
    print(f"  dloss/d{name:<2} {t.grad.item():>7.1f}")

forward    z1=  6.0  z2= 10.0  a= 10.0  loss=  4.0

  dloss/da     -4.0
  dloss/dz2    -4.0
  dloss/dz1    -4.0
  dloss/db     -4.0
  dloss/dw    -12.0
  dloss/dx     -8.0


## Reading gradients out of `.grad`

It approximated nothing. It ran the chain rule you just ran, on the graph it
recorded during the forward pass, and every number matches the hand derivation.

- `w.grad = -12` is $\partial \ell / \partial w$. SGD will use it
- `b.grad = -4` is $\partial \ell / \partial b$. SGD will use it
- `x.grad = -8` is $\partial \ell / \partial x$. It got computed on the way
  past and we throw it away, because the input is not a parameter

Finite differences would have needed one forward pass per scalar. This needed
one forward and one backward, for all of them at once.

::: {.callout-tip}
## Where $\partial \ell / \partial x$ stops being garbage
Adversarial examples, saliency maps and style transfer all optimize the
**input** while holding the weights fixed. Same machinery, different variable.
:::

## Scalar classification graph

[![](images/scalar-backpropagation-classification-simplified.png){width=714 .dw68}](images/scalar-backpropagation-classification-simplified.png){target="_blank" .zoom}

Same shape, different head: sigmoid instead of ReLU, binary cross-entropy
instead of squared error. Take $x = 2$, $w = 0.5$, $b = -2.1$, and $y = 0$.

$$z_2 = -1.1, \qquad
  \hat{y} = \sigma(z_2) = 0.2497, \qquad
  \ell = -\log(1 - \hat{y}) = 0.2873$$

## Scalar classification graph, both passes

[![](images/scalar-backpropagation-classification.png){width=1050 .dw95}](images/scalar-backpropagation-classification.png){target="_blank" .zoom}

The same expansion as the regression case, now with a sigmoid and binary cross
entropy. The chain is longer because the loss is written out as
$\hat{y} = 1/a_3$ rather than folded into one node.

## Derivative of binary cross-entropy {.smaller}

[![](images/2024-03-12-20-12-43.png){width=714 .dw68}](images/2024-03-12-20-12-43.png){target="_blank" .zoom}

Worth keeping for the two class cases at the middle of the slide: the same
formula gives $+1.33$ when $y = 0$ and $-4$ when $y = 1$, from the identical
prediction $\hat{y} = 0.25$. The sign is what tells the weight which way to
move.

## Classification in PyTorch

Watch $\partial \ell / \partial z_2$ in the output and compare it to
$\hat{y} - y$.

In [2]:
#| echo: true
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([-2.1], requires_grad=True)
y = torch.tensor([0.0])

z1 = w * x;                    z1.retain_grad()
z2 = z1 + b;                   z2.retain_grad()
y_hat = torch.sigmoid(z2);     y_hat.retain_grad()
loss = nn.BCELoss()(y_hat, y)

loss.backward()

print(f"z2 = {z2.item():.4f}   y_hat = {y_hat.item():.4f}   "
      f"loss = {loss.item():.4f}")
print()
print(f"  dloss/dz2  {z2.grad.item():.4f}")
print(f"  y_hat - y  {(y_hat - y).item():.4f}   <- the same number")
print()
for name, t in [("w", w), ("b", b), ("x", x)]:
    print(f"  dloss/d{name}   {t.grad.item():>7.4f}")

z2 = -1.1000   y_hat = 0.2497   loss = 0.2873

  dloss/dz2  0.2497
  y_hat - y  0.2497   <- the same number

  dloss/dw    0.4995
  dloss/db    0.2497
  dloss/dx    0.1249


## Gradient of cross-entropy with a sigmoid output

The two ugly derivatives cancel. For binary cross-entropy on a sigmoid output:

$$\frac{\partial \ell}{\partial \hat{y}}
  = \frac{\hat{y} - y}{\hat{y}(1 - \hat{y})},
  \qquad
  \frac{\partial \hat{y}}{\partial z}
  = \hat{y}(1 - \hat{y})
  \quad\Longrightarrow\quad
  \frac{\partial \ell}{\partial z} = \hat{y} - y$$

The gradient entering the linear part is just **the prediction minus the
truth**. Squared error on a linear output gives the same form. It is why these
loss and activation pairings are the defaults, and it is the same algebra Week 1
met in logistic regression.

This is also why `nn.CrossEntropyLoss` takes logits and applies
$\log\mathrm{softmax}$ itself: doing them together is both simpler and
numerically stabler than doing them apart.

## Forward mode and reverse mode

[![](images/fig-forward-vs-reverse.png){width=1100 .dw85}](images/fig-forward-vs-reverse.png){target="_blank" .zoom}

Autodiff has two directions, and the choice is not stylistic.

- **Forward mode** propagates a derivative from one input forward. To get all
  $n$ parameter gradients you sweep $n$ times
- **Reverse mode** propagates a derivative from one output backward. One sweep
  gives you every input gradient

## Choosing forward or reverse mode

The cost of each mode is set by the **shape** of the function, not by the
architecture:

| | Sweeps needed | Good when |
|---|---|---|
| Forward mode | one per **input** | few inputs, many outputs |
| Reverse mode | one per **output** | many inputs, **one** output |

A neural network maps millions of parameters to a **single scalar cost**. That
is the best possible case for reverse mode and the worst possible case for
forward mode.

If the cost were a vector of a million values, forward mode would win and deep
learning would look very different. It is a scalar because we chose to reduce
the batch to a mean, which brings us back to loss versus cost.

# From a scalar to a layer

- The same three rules, now on matrices
- What gets stored, and what does not

## Backward pass of a linear layer

Week 2: a layer with $\mathbf{W}$ of shape `(units, inputs)` computes
$\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}$.

Given $\partial \mathcal{L} / \partial \mathbf{z}$ arriving from the next
layer, the layer owes three things:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}}
  = \frac{\partial \mathcal{L}}{\partial \mathbf{z}}\, \mathbf{x}^{\top},
  \qquad
  \frac{\partial \mathcal{L}}{\partial \mathbf{b}}
  = \frac{\partial \mathcal{L}}{\partial \mathbf{z}},
  \qquad
  \frac{\partial \mathcal{L}}{\partial \mathbf{x}}
  = \mathbf{W}^{\top} \frac{\partial \mathcal{L}}{\partial \mathbf{z}}$$

The first two are what the optimizer updates. The third is the upstream gradient
for the layer **behind** this one, which is what makes the pass propagate.

## Shape check on the layer gradients

The formulas are memorable because the shapes force them. A layer with 3 units
and 2 inputs, on a single example:

| Object | Shape | Why |
|---|---|---|
| $\mathbf{W}$ | (3, 2) | units by inputs |
| $\mathbf{x}$ | (2,) | what came in |
| $\partial \mathcal{L} / \partial \mathbf{z}$ | (3,) | one per unit |
| $\partial \mathcal{L} / \partial \mathbf{W}$ | (3, 2) | (3,) outer (2,) |
| $\partial \mathcal{L} / \partial \mathbf{x}$ | (2,) | (2, 3) times (3,) |

Only one arrangement of $\mathbf{W}$, $\mathbf{W}^{\top}$ and the incoming
gradient produces each required shape. **If the shapes work, the formula is
almost certainly right**, and that is the debugging trick to remember.

## Layer backward pass, verified against autograd {.codetight}

Lab 2's 2-3-2-1 network, same seed. The whole backward pass written out in
NumPy, then checked against `loss.backward()`. Source code for your reference;
the numbers are the point.

In [3]:
#| echo: true
import numpy as np

rng = np.random.default_rng(6600)
W1, b1 = rng.normal(0, 0.8, (3, 2)), np.zeros(3)
W2, b2 = rng.normal(0, 0.8, (2, 3)), np.zeros(2)
W3, b3 = rng.normal(0, 0.8, (1, 2)), np.zeros(1)
X, Y = rng.normal(0, 1, (5, 2)), rng.normal(0, 1, (5, 1))
N = X.shape[0]

# forward, keeping every activation because the backward pass needs them
z1 = X @ W1.T + b1;   a1 = np.tanh(z1)
z2 = a1 @ W2.T + b2;  a2 = np.tanh(z2)
z3 = a2 @ W3.T + b3
cost = float(np.mean((z3 - Y) ** 2))

# backward, right to left: seed, then (local gradient) x (upstream gradient)
dz3 = 2 * (z3 - Y) / N                     # d cost / d z3
gW3, gb3 = dz3.T @ a2, dz3.sum(0)          # this layer's parameters
dz2 = (dz3 @ W3) * (1 - a2 ** 2)           # through W3, then through tanh
gW2, gb2 = dz2.T @ a1, dz2.sum(0)
dz1 = (dz2 @ W2) * (1 - a1 ** 2)
gW1, gb1 = dz1.T @ X, dz1.sum(0)

# the same thing, asked of autograd
model = nn.Sequential(nn.Linear(2, 3), nn.Tanh(),
                      nn.Linear(3, 2), nn.Tanh(),
                      nn.Linear(2, 1)).double()
with torch.no_grad():
    for layer, (W, b) in zip([model[0], model[2], model[4]],
                             [(W1, b1), (W2, b2), (W3, b3)]):
        layer.weight.copy_(torch.from_numpy(W))
        layer.bias.copy_(torch.from_numpy(b))
torch_cost = nn.MSELoss()(model(torch.from_numpy(X)), torch.from_numpy(Y))
torch_cost.backward()

print(f"cost   by hand {cost:.6f}   autograd {float(torch_cost):.6f}\n")
print(f"{'':>4}{'shape':>9}{'max |by hand - autograd|':>28}")
for name, mine, p in [("W1", gW1, model[0].weight), ("b1", gb1, model[0].bias),
                      ("W2", gW2, model[2].weight), ("b2", gb2, model[2].bias),
                      ("W3", gW3, model[4].weight), ("b3", gb3, model[4].bias)]:
    print(f"{name:>4}{str(mine.shape):>9}"
          f"{np.abs(mine - p.grad.numpy()).max():>28.2e}")

cost   by hand 2.863851   autograd 2.863851

        shape    max |by hand - autograd|
  W1   (3, 2)                    4.44e-16
  b1     (3,)                    2.22e-16
  W2   (2, 3)                    2.22e-16
  b2     (2,)                    2.22e-16
  W3   (1, 2)                    3.33e-16
  b3     (1,)                    2.22e-16


## The three rules that reproduced autograd

Twelve lines of NumPy reproduced `loss.backward()` to floating-point noise, on a
three-layer network, using nothing but:

- $\partial \mathcal{L}/\partial \mathbf{W} = \partial \mathcal{L}/\partial \mathbf{z} \cdot \mathbf{x}^{\top}$
- $\partial \mathcal{L}/\partial \mathbf{x} = \mathbf{W}^{\top} \cdot \partial \mathcal{L}/\partial \mathbf{z}$
- $\tanh'(z) = 1 - \tanh^2(z)$, which reuses the cached activation

Autograd is not doing anything you cannot do. It is doing it for an arbitrary
graph, without you writing the six lines per layer, and without getting a
transpose wrong at 2 AM.

## ANN matrix notation {.smaller}

The same network, written out in full. Worth having on hand once; not worth
reading aloud.

:::: {.columns}
::: {.column width="55%"}
[![](images/ann-matrix-notation.png){width=450 .dh460}](images/ann-matrix-notation.png){target="_blank" .zoom}
:::
::: {.column width="45%"}
Notation to keep straight:

- $\mathbf{W}^{(\ell)}$ is layer $\ell$'s weights, `(units, inputs)`
- $\mathbf{z}^{(\ell)}$ is the pre-activation
- $\mathbf{a}^{(\ell)} = g(\mathbf{z}^{(\ell)})$ is the activation
- $\mathbf{a}^{(0)} = \mathbf{x}$ is the input
- The superscript is the layer, the subscript is the unit

Every quantity in the backward pass has a matching shape on this diagram.
:::
::::

## ANN forward and backward, worked {.smaller}

:::: {.columns}
::: {.column width="50%"}
**Forward**

[![](images/ann-forward-regression.png){width=510 .dh420}](images/ann-forward-regression.png){target="_blank" .zoom}
:::
::: {.column width="50%"}
**Backward**

[![](images/ann-backward-regression.png){width=543 .dh420}](images/ann-backward-regression.png){target="_blank" .zoom}
:::
::::

Left to right, then right to left, on the same network. These are on the
handout at full size; the point here is the symmetry, not the entries.

## Jacobian memory cost

The full Jacobian of a layer holds the derivative of every output coordinate
with respect to every input coordinate. For a batch of 128, 128 features in and
256 units out, in float32:

$$128 \times 128 \times 128 \times 256 \times 4 \text{ bytes}
  \approx 6.5 \text{ GB}$$

For **one layer**. The two gradients training actually needs,
$\partial \mathcal{L}/\partial \mathbf{W}$ and
$\partial \mathcal{L}/\partial \mathbf{x}$, come to about **198 KB**.

Reverse mode never forms the Jacobian. It only ever computes
Jacobian-times-vector, which is the matrix multiply on the previous slides. That
is the difference between a model that fits in memory and one that does not.

## One entry of the weight gradient {.smaller}

[![](images/matrix-multiplication-differentiation-w11.png){width=630 .dw60}](images/matrix-multiplication-differentiation-w11.png){target="_blank" .zoom}

Expanding $\partial J / \partial w_{1,1}$ gives eight terms and **six are
zero**, leaving a column of $\mathbf{X}$ dotted with a column of
$\nabla_{\mathbf{Z}} J$. That pattern, over every entry, is exactly
$\nabla_{\mathbf{W}} J = \mathbf{X}^{\top} \nabla_{\mathbf{Z}} J$.

## Backward pass in node form: input to ReLU {.smaller}

[![](images/ann-backward-node-left.png){width=1050 .dw95}](images/ann-backward-node-left.png){target="_blank" .zoom}

One column per node, with the matrix each node hands backward written out. At
the multiply node, the two familiar rules:
$\nabla_{\mathbf{X}} J = \nabla_{\mathbf{Z}} J \mathbf{W}^{\top}$ and
$\nabla_{\mathbf{W}} J = \mathbf{X}^{\top} \nabla_{\mathbf{Z}} J$. At the
ReLU, a gate: pass the entry through where $z > 0$, otherwise zero.

## Backward pass in node form: output layer to cost {.smaller}

[![](images/ann-backward-node-right.png){width=1050 .dw95}](images/ann-backward-node-right.png){target="_blank" .zoom}

The right half of the same strip. The linear output layer has derivative 1, so
it passes its gradient straight through, and the cost node starts the whole
chain with $\nabla_{\mathbf{A}} J = \frac{2}{m}(\hat{\mathbf{Y}} -
\mathbf{Y})$.

## Exploding and vanishing gradients

[![](images/exploding-vanishing-gradient.png){width=735 .dw70}](images/exploding-vanishing-gradient.png){target="_blank" .zoom}

The gradient is a product of one factor per layer, so it compounds as it travels
right to left. Consistently large factors make it **explode** before it reaches
the early layers; consistently small ones make it **vanish**.

Either way the layers nearest the input are the ones that suffer, which is
exactly the opposite of what you want, since they set the features every later
layer builds on.

## Gradient magnitude by layer, measured {.smaller}

[![](images/fig-gradient-decay.png){width=1050 .dw85}](images/fig-gradient-decay.png){target="_blank" .zoom}

Twenty layers, real gradients read out of autograd. The chain rule
**multiplies**, so a local derivative consistently below 1 drives the product to
zero and one consistently above 1 drives it to infinity.

Sigmoid's derivative peaks at $0.25$. Twenty of those in a row is $10^{-13}$,
which is what the blue line shows: **the layers nearest the input get no signal
at all.** Scaling the weights up 60% sends the same architecture to $10^{7}$.

## Causes and cures {.smaller}

:::: {.columns .contrast}
::: {.column width="48%" .col-no}
### Exploding
Local derivatives above 1, compounding.

- Weight initialization too large, or a weight update that overshot
- **Gradient clipping** caps the norm and is very effective
- **Weight regularization**, $\mathcal{L} = \text{error} + \lambda \sum w_i^2$
- **Batch normalization**
:::
::: {.column width="48%" .col-yes}
### Vanishing
Local derivatives near 0, compounding.

- Saturating activations, and excessive depth
- **ReLU** instead of sigmoid, since its derivative is exactly 1 where it is on
- **Skip connections**, as in ResNet, hand the input forward past layers
- **Better initialization**, Kaiming or Xavier
:::
::::

This is a **training** problem, not an autodiff problem. Autograd will hand you a
gradient of $10^{15}$ without complaint. **Every cure listed here is Week 4.**

# PyTorch autograd

- A tensor is an array that remembers where it came from
- Everything else in this section is vocabulary

## What a tensor is

A **tensor** is an $n$-dimensional array. **Rank**, or `ndim`, is the number of
axes, meaning the number of indices needed to reach one element.

| Rank | Shape | Name |
|---|---|---|
| 0 | `[]` | scalar |
| 1 | `[n]` | vector |
| 2 | `[m, n]` | matrix |
| $k$ | `[d1, ..., dk]` | tensor |

A PyTorch tensor is a NumPy array plus three things Lab 2 did not have:

- A **device**, CPU or CUDA or MPS
- A **`requires_grad`** flag
- A recorded **graph**, so it knows which operation produced it

## Tensors and NumPy arrays

The arithmetic is the same. `torch.from_numpy` even shares memory on CPU, so
writing through one view changes the other.

In [4]:
#| echo: true
A_np = np.arange(6).reshape(2, 3)
A = torch.from_numpy(A_np)          # shares storage with A_np on CPU

print(f"numpy  {A_np.shape}  torch  {tuple(A.shape)}  ndim {A.ndim}  "
      f"numel {A.numel()}")

A_np[0, 0] = 99
print(f"wrote through the numpy view; torch sees {A[0, 0].item()}")

x = torch.tensor([[0.7, -1.2]])                  # one batched example, (1, 2)
W = torch.randn(3, 2, requires_grad=True)        # (units, inputs), as in Lab 2
b = torch.zeros(3, requires_grad=True)
z = x @ W.T + b                                  # the batched layer from Lab 2

print(f"\nx {tuple(x.shape)} @ W.T {tuple(W.T.shape)} + b {tuple(b.shape)} "
      f"-> z {tuple(z.shape)}")
print(f"z was produced by {type(z.grad_fn).__name__}, "
      f"which is the graph autograd recorded")

numpy  (2, 3)  torch  (2, 3)  ndim 2  numel 6
wrote through the numpy view; torch sees 99

x (1, 2) @ W.T (2, 3) + b (3,) -> z (1, 3)
z was produced by AddBackward0, which is the graph autograd recorded


## dtype and device {.smaller}

Two attributes that cause most first-week PyTorch errors.

:::: {.columns}
::: {.column width="52%"}
**dtype.** Memory is `numel` times bytes per element.

| dtype | Bytes | Used for |
|---|---|---|
| `float32` | 4 | the default for training |
| `float64` | 8 | precision, and slower |
| `float16` | 2 | fast GPU training |
| `bfloat16` | 2 | wider exponent, stabler |
| `int64` | 8 | indices, class labels |
:::
::: {.column width="48%"}
**device.** Tensors must be on the same device to interact.

```python
torch.cuda.is_available()
x = x.to("cuda")
```

`RuntimeError: Expected all tensors to be on the same device` is the error you
will hit in Lab 3. The fix is always to move one of them.

`nn.CrossEntropyLoss` wants labels as `int64`. That is the other one.
:::
::::

## `requires_grad` and the tape

Three rules that explain most of autograd's behaviour:

1. An operation is recorded if **any** input has `requires_grad=True`
2. `.backward()` is called on a **scalar**, and it fills `.grad` on every leaf
   tensor that requested gradients
3. Intermediate results do not keep their gradient unless you ask with
   `retain_grad()`

To stop recording, either wrap the block in `torch.no_grad()` or call
`.detach()`. Do this for evaluation and inference: it is faster and it does not
build a graph you are about to throw away.

```python
with torch.no_grad():
    val_loss = criterion(model(x_val), y_val)
```

## Gradients accumulate

`.grad` is a **running sum**, for the reason we saw at the fork. Here is the
failure mode rather than a warning about it.

In [5]:
#| echo: true
w = torch.tensor([2.0], requires_grad=True)
x, y = torch.tensor([3.0]), torch.tensor([12.0])

for call in (1, 2, 3):
    loss = (w * x - y) ** 2
    loss.backward()
    print(f"backward call {call}:  w.grad = {w.grad.item():>7.1f}"
          f"   ({call}x the true gradient)")

w.grad.zero_()
loss = (w * x - y) ** 2
loss.backward()
print(f"\nafter zero_():   w.grad = {w.grad.item():>7.1f}   (correct again)")

backward call 1:  w.grad =   -36.0   (1x the true gradient)
backward call 2:  w.grad =   -72.0   (2x the true gradient)
backward call 3:  w.grad =  -108.0   (3x the true gradient)

after zero_():   w.grad =   -36.0   (correct again)


## `zero_grad` and gradient accumulation

Every training step is these four lines, and the order matters:

```python
optimizer.zero_grad()    # clear the running sum from last step
loss = criterion(model(x), y)
loss.backward()          # write .grad on every parameter
optimizer.step()         # theta <- theta - lr * theta.grad
```

Forget `zero_grad()` and step 3 adds to whatever was already there. You are not
descending on the current gradient; you are descending on the sum of every
gradient since the model was built. The loss usually diverges, and it looks
exactly like a learning rate that is too high.

::: {.callout-note}
## The accumulation is a feature too
Deliberately skipping `zero_grad()` for a few mini-batches simulates a larger
batch than fits in memory. That is **gradient accumulation**, and it is a real
technique. It is only a bug when it is an accident.
:::

## `nn.Linear` is Lab 2's layer

`nn.Linear(in_features, out_features)` stores $\mathbf{W}$ as
`(out_features, in_features)` and $\mathbf{b}$ as `(out_features,)`. That is
the `(units, inputs)` convention from Week 2, unchanged.

In [6]:
#| echo: true
torch.manual_seed(6600)
layer = nn.Linear(2, 3)
x = torch.randn(5, 2)                       # five examples, two features

print(f"layer.weight {tuple(layer.weight.shape)}   "
      f"layer.bias {tuple(layer.bias.shape)}")
print(f"x {tuple(x.shape)}  ->  layer(x) {tuple(layer(x).shape)}")

# the same arithmetic, written out
by_hand = x @ layer.weight.T + layer.bias
print(f"\nmax |layer(x) - (x @ W.T + b)| = "
      f"{(layer(x) - by_hand).abs().max().item():.2e}")
print("nn.Linear is that line, plus the gradient tape")

layer.weight (3, 2)   layer.bias (3,)
x (5, 2)  ->  layer(x) (5, 3)

max |layer(x) - (x @ W.T + b)| = 0.00e+00
nn.Linear is that line, plus the gradient tape


## `nn.Module`

A model is a class with two jobs: declare the layers in `__init__`, and write
the forward pass in `forward`. Subclassing `nn.Module` is what registers the
parameters so the optimizer, `.to(device)` and saving all find them.

In [7]:
#| echo: true
class MLP(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.hidden = nn.Linear(1, hidden)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        return self.out(torch.tanh(self.hidden(x)))


model = MLP()
print(model)
print()
for name, p in model.named_parameters():
    print(f"  {name:<14} {str(tuple(p.shape)):>10}  {p.numel():>3} "
          f"parameter{'s' if p.numel() != 1 else ''}")
print(f"\ntotal: {sum(p.numel() for p in model.parameters())} "
      f"(Lab 2's network, exactly)")

MLP(
  (hidden): Linear(in_features=1, out_features=16, bias=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

  hidden.weight     (16, 1)   16 parameters
  hidden.bias         (16,)   16 parameters
  out.weight        (1, 16)   16 parameters
  out.bias             (1,)    1 parameter

total: 49 (Lab 2's network, exactly)


## `model(x)` versus `model.forward(x)`

Call `model(x)`, never `model.forward(x)`. The first runs registered hooks and
respects train and eval mode; the second skips them.

- `nn.Sequential` is the same thing when the graph is a straight line, which is
  all of today
- `model.train()` and `model.eval()` change the behaviour of dropout and batch
  normalization. Neither is in today's model, so neither matters yet, and both
  will bite you in Week 4 if you forget them
- `model.parameters()` is what you hand the optimizer

```python
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
```

## Loss functions in PyTorch

| Task | Final layer | PyTorch loss |
|---|---|---|
| Regression | linear, 1 unit | `nn.MSELoss` |
| Binary classification | linear, 1 unit | `nn.BCEWithLogitsLoss` |
| Multi-class, $C$ classes | linear, $C$ units | `nn.CrossEntropyLoss` |

Note what is **not** in that middle column: no sigmoid, and no softmax.

`nn.BCELoss` exists and takes probabilities, so it needs a sigmoid in the model.
Prefer `BCEWithLogitsLoss`, which folds the sigmoid in and is numerically
stabler for the reason on the cancellation slide.

## Logits and `CrossEntropyLoss`

In Keras you apply softmax in the model. **In PyTorch you do not.**
`nn.CrossEntropyLoss` applies $\log\mathrm{softmax}$ internally and then takes
the negative log-likelihood:

$$\mathcal{L}(z, y) = -\log \frac{e^{z_y}}{\sum_j e^{z_j}}$$

So classification models in this course return **logits**, raw scores of shape
`(N, C)`, and the targets are **integer class labels** of shape `(N,)` with
dtype `int64`. Not one-hot.

Apply softmax in the model as well and you have applied it twice: the gradients
degrade and the numerics get worse. It is a quiet bug, because the model still
trains, just badly.

# The first training loop

- Lab 2's network, Lab 2's data, Lab 2's initialization
- One backward pass per step instead of fifty forward passes

## Full batch, mini-batch, stochastic {.smaller}

One **optimizer step** is one parameter update. How much data goes into it has a
name. For $N = 100$ training examples:

| Paradigm | Batch size | Updates per epoch | Gradient noise |
|---|---|---|---|
| Full batch | 100 | 1 | none |
| Stochastic | 1 | 100 | very high |
| Mini-batch | 25 | 4 | moderate |

An **epoch** is one pass through the whole training set.

Mini-batch is the standard, and the noise turns out to be useful rather than
merely tolerable. Today's demo is **full batch** on 200 points, so there is one
update per epoch and the loss curve is smooth. `DataLoader` and real mini-batches
are Week 4.

## The loop

These five lines will not change for the rest of the semester:

```python
for step in range(STEPS):
    optimizer.zero_grad()               # clear the running sum
    loss = criterion(model(x), y)        # forward, and record
    loss.backward()                      # backward, fill every .grad
    optimizer.step()                     # theta <- theta - lr * grad
```

Everything else is which model, which loss, which optimizer, and what data comes
in. The optimizer is plain **SGD** today, the same update rule from Week 1:

$$\theta^{(t+1)} = \theta^{(t)} - \eta\, \nabla_{\theta} \mathcal{L}\big(\theta^{(t)}\big)$$

Adam, schedules and momentum are Week 4. Today we only want the gradient to be
correct and cheap.

## Lab 2's network, trained with autograd {.codetight}

Same seed, same weights, same 400 steps, same learning rate. Source code for
your reference. The printed losses should match Lab 2 to four decimals; the work
per step should not.

In [8]:
#| echo: true
import time

SEED, HIDDEN, STEPS, LR = 6600, 16, 400, 0.1
SHAPES = [("W", (HIDDEN, 1)), ("b", (HIDDEN,)), ("W", (1, HIDDEN)), ("b", (1,))]

# Lab 2's data and initialization, reproduced exactly: two generators, both
# seeded with SEED, so neither advances the other's stream.
rng_data, rng_init = np.random.default_rng(SEED), np.random.default_rng(SEED)
x_np = np.linspace(-1, 1, 200).reshape(-1, 1)
y_np = np.sin(3.0 * x_np.ravel()) + rng_data.normal(0, 0.10, 200)

theta, i = np.zeros(sum(int(np.prod(s)) for _, s in SHAPES)), 0
for kind, shape in SHAPES:
    n = int(np.prod(shape))
    theta[i:i + n] = (rng_init.normal(0, 1 / np.sqrt(shape[1]), n)
                      if kind == "W" else 0.0)
    i += n

model = nn.Sequential(nn.Linear(1, HIDDEN), nn.Tanh(),
                      nn.Linear(HIDDEN, 1)).double()
with torch.no_grad():
    model[0].weight.copy_(torch.from_numpy(theta[:HIDDEN].reshape(HIDDEN, 1)))
    model[0].bias.copy_(torch.from_numpy(theta[HIDDEN:2 * HIDDEN]))
    model[2].weight.copy_(torch.from_numpy(
        theta[2 * HIDDEN:3 * HIDDEN].reshape(1, HIDDEN)))
    model[2].bias.copy_(torch.from_numpy(theta[-1:]))

x = torch.from_numpy(x_np)
y = torch.from_numpy(y_np).unsqueeze(1)
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
criterion = nn.MSELoss()

start = float(criterion(model(x), y))
t0 = time.perf_counter()
for step in range(STEPS):
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
elapsed = time.perf_counter() - t0
end = float(criterion(model(x), y))
n_params = sum(p.numel() for p in model.parameters())

print(f"parameters      {n_params}")
print(f"cost at start   {start:.4f}      (Lab 2: 1.3256)")
print(f"cost at end     {end:.4f}      (Lab 2: 0.0190)")
print(f"wall time       {elapsed:.2f} s for {STEPS} steps")
print(f"work per step   1 forward + 1 backward, not {n_params + 1} forwards")

parameters      49
cost at start   1.3256      (Lab 2: 1.3256)
cost at end     0.0190      (Lab 2: 0.0190)
wall time       0.15 s for 400 steps
work per step   1 forward + 1 backward, not 50 forwards


## Loss curves, autograd and finite differences

[![](images/demo-autograd-training.gif){width=880 .dw85}](images/demo-autograd-training.gif){target="_blank" .zoom}

The grey curve is Lab 2's finite-difference run. The red curve is today's. They
agree to $3.8 \times 10^{-7}$ over all 400 steps, because both are estimating
the **same gradient of the same cost on the same data**.

Finite differences were never wrong. They were just expensive.

## Finite differences and autograd, side by side

| | Lab 2 | Today |
|---|---|---|
| Gradient from | finite differences | reverse-mode autodiff |
| Cost at start | 1.3256 | 1.3256 |
| Cost at end | 0.0190 | 0.0190 |
| Work per step | 50 forward passes | 1 forward + 1 backward |
| Work for 400 steps | 20,001 forward passes | 400 backward passes |
| Exact? | no, $O(\varepsilon)$ error | yes, to floating point |

Same descent, 50x less work, and no $\varepsilon$ to tune. At ResNet-50 scale
the ratio is not 50, it is 25 million.

The gradient has stopped being the expensive part. Everything that can still go
wrong is now about the **optimizer**, the **data**, and the **loss surface**.

## Scope: today and Week 4 {.smaller}

Left out on purpose, and all of it lands next week:

:::: {.columns .contrast}
::: {.column width="48%" .col-yes}
### Today
- Why finite differences do not scale
- The chain rule on a graph
- Reverse-mode autodiff, by hand and in PyTorch
- Tensors, `.backward()`, `.grad`, `zero_grad`
- `nn.Linear`, `nn.Module`, SGD
- One 49-parameter training loop
:::
::: {.column width="48%" .col-no}
### Week 4
- Adam, RMSProp, momentum, weight decay
- Dropout, batch norm, skip connections
- Learning-rate schedules, gradient clipping
- Initialization as a decision you make
- `Dataset`, `DataLoader`, real mini-batches
- A training run worth the name
:::
::::

You should leave able to read any PyTorch training loop and say which line is
the backward pass. You should not leave an optimizer expert.

# Closing

- Lab 3
- Quiz 3 study guide
- Supplemental content
- Quiz 2, right now

## Lab 3 {.smaller}

**Released today. Due Wednesday Sep 16, 11:59 PM ET.**

Lab 2's network, in PyTorch:

1. Wrap the $1 \to 16 \to 1$ model as an `nn.Module`
2. Delete `finite_difference_gradient`; call `loss.backward()` instead
3. Confirm your loss curve matches the one you produced last week
4. Break it on purpose: run the loop once with `zero_grad()` removed, and
   report what happened
5. Hand-derive the gradient for one weight and check it against `.grad`

Same rules as Labs 1 and 2: graded for **completion**, work together if you
like, solutions posted after the deadline.

::: {.callout-note}
## Why this order
Last week you paid 20,001 forward passes for 400 steps. This week the same fit
is one `loss.backward()` per step. The payoff only lands if you felt the bill
first.
:::

## Quiz 3 study guide {.smaller}

**Next week, end of class.** Closed-book, no notes. A formula sheet comes with
the quiz.

Everything on it comes from today and Lab 3:

1. **Why finite differences do not scale**, and what reverse-mode autodiff
   computes instead
2. **The chain rule on a graph**: the add node, the multiply node, the max node,
   and why gradients sum over paths
3. **`.backward()`, `.grad`, `zero_grad()`**: what each does, and what breaks if
   you skip the third
4. **Reverse mode versus forward mode**, and why a scalar cost decides it
5. **`nn.Linear` shapes**: `(out_features, in_features)`, matching Lab 2
6. **The training loop**, in order: zero, forward, loss, backward, step
7. **Loss versus cost**, and which one `.backward()` is called on
8. **From Lab 3**: what changed when autograd replaced finite differences, and
   what happened when you removed `zero_grad()`. Explanations, not numbers

## Supplemental content {.smaller}

These are **optional**. No graded work assumes you read them. They go deeper
than we had time for, and the first two are the source notebooks this deck was
built from.

:::: {.columns}
::: {.column width="50%"}
**Autodiff and gradients**

- [Automatic differentiation](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/backprop/automatic-differentiation/automatic-differentiation.html)
- [Matrix-multiplication differentiation](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/backprop/matrix-multiplication-differentiation/matrix-multiplication-differentiaion.html)
- [Gradient descent](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/fundamentals/gradient-descent/notes.html)
- [Backprop basics, video](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/backprop/backprop-basics-video/notes.html)
:::
::: {.column width="50%"}
**PyTorch**

- [Tensors](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/pytorch/torch-tensors/notes.html)
- [Models](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/pytorch/torch-models/torch-models.html)
- [Data](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/pytorch/torch-data/torch-data.html)
:::
::::

<sup>All of these live on the
[centralized lecture content](https://jfh.georgetown.domains/centralized-lecture-content/)
site. The autodiff page also carries a from-scratch `Node` class that implements
today's graph, topological sort and backward pass in about 80 lines of NumPy.
The PyTorch pages carry the Week 4 material as well, so expect to see them
again.</sup>

## Wrap-up {.smaller}

Where we started: 20,001 forward passes for one decent fit. Where we are:

- A **computational graph** is the forward pass, recorded. Values at the nodes,
  operations on the edges
- **Backpropagation** is the chain rule on that graph, right to left, multiplying
  local derivatives and **adding where paths meet**
- **Reverse mode** is the right direction because the cost is a **scalar**
- A **layer's** backward pass is two matrix multiplies. Nobody forms a Jacobian
- **Autograd** implements all of it. `.backward()` is the backward pass;
  `zero_grad()` clears the running sum
- **`nn.Linear`** is Lab 2's layer and **`nn.Module`** is a forward pass with
  parameters attached
- The **training loop** is five lines, and the interesting one is `optimizer`

::: {.callout-note}
## Next week
**Training deep neural networks.** Adam, regularization, schedules,
initialization, mini-batches, and all the reasons a correct gradient is not the
same thing as a training run that works.
:::

**Before Thursday:** Lab 3, due Wednesday. And claim a Spotlight slot if you
have not.

## Quiz 2

Closed-book, no notes. A formula sheet comes with it.

Covers **Week 2 and Lab 2**:

- Layer shapes and parameter counts
- The forward pass, single example and batched
- Why nonlinearities exist, and what stacked linear layers collapse to
- Activations: sigmoid, tanh, ReLU, softmax
- Depth versus width
- Why finite-difference gradients do not scale

Twenty-five points. Show your work; an answer with no reasoning earns partial
credit at best.